In [6]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from fractions import Fraction
import matplotlib.pyplot as plt

In [7]:
# -----------------------------
# PARAMETERS
# -----------------------------
N = 21        # Number to factor
a = 2         # coprime with N
n_count = 5   # counting qubits
n_work = 5    # work qubits (enough to store values mod N)


In [8]:
# -----------------------------
# QFT & inverse QFT
# -----------------------------
def qft(n):
    qc = QuantumCircuit(n)
    for j in range(n):
        qc.h(j)
        for k in range(j+1, n):
            qc.cp(np.pi/(2**(k-j)), k, j)
    return qc

def qft_dagger(n):
    qc = QuantumCircuit(n)
    for j in reversed(range(n)):
        for k in reversed(range(j+1, n)):
            qc.cp(-np.pi/(2**(k-j)), k, j)
        qc.h(j)
    return qc


In [9]:
# -----------------------------
# Modular addition (Fourier-based)
# -----------------------------
def add_const(qc, a, n):
    """Add constant 'a' in Fourier basis"""
    for j in range(n):
        angle = (a * np.pi) / (2**j)
        qc.p(angle, j)

def modular_addition(a, N, n):
    """|x> -> |x + a mod N> with 1 ancilla"""
    qc = QuantumCircuit(n+1)

    qc.append(qft(n), range(n))
    add_const(qc, a, n)
    add_const(qc, -N, n)
    qc.append(qft_dagger(n), range(n))

    qc.cx(n-1, n)          # MSB -> ancilla
    qc.append(qft(n), range(n))
    add_const(qc, N, n)
    qc.append(qft_dagger(n), range(n))

    return qc

In [10]:
# -----------------------------
# Controlled modular multiplication (educational)
# -----------------------------
def controlled_mult(a, N, n):
    """
    Simplified controlled multiplication: |y> -> |a*y mod N> controlled by 1 qubit
    Here we just illustrate the structure for small N
    """
    qc = QuantumCircuit(n, name=f"{a}*y mod {N}")
    # For simplicity, we just map 0..N-1 manually (works for small N)
    dim = 2**n
    U = np.zeros((dim, dim))
    for y in range(dim):
        if y < N:
            new_y = (a*y) % N
        else:
            new_y = y
        U[new_y, y] = 1
    from qiskit.circuit.library import UnitaryGate
    gate = UnitaryGate(U)
    return gate.control()

In [12]:
# -----------------------------
# Modular exponentiation (controlled)
# -----------------------------
def modular_exponentiation(a, N, n_count, n_work):
    qc = QuantumCircuit(n_count + n_work, n_count)

    # Counting qubits in superposition
    for q in range(n_count):
        qc.h(q)

    # Work register initialized to |1>
    qc.x(n_count)

    # Apply controlled modular multiplications
    for i in range(n_count):
        # Each controlled multiplication corresponds to a^(2^i)
        mult_gate = controlled_mult(pow(a, 2**i, N), N, n_work)
        qc.append(mult_gate, [i] + [j+n_count for j in range(n_work)])

    # Inverse QFT
    qc.append(qft_dagger(n_count), range(n_count))

    # Measure counting qubits
    qc.measure(range(n_count), range(n_count))

    return qc


In [13]:
# -----------------------------
# Run the simulation
# -----------------------------
qc = modular_exponentiation(a, N, n_count, n_work)
simulator = AerSimulator()
tcirc = transpile(qc, simulator)
job = simulator.run(tcirc, shots=1024)
result = job.result()
counts = result.get_counts()
print("Counts:", counts)

# Plot histogram
plot_histogram(counts)
plt.show()

# -----------------------------
# Classical post-processing: phase -> period
# -----------------------------
print("\nPhase estimation:")
for output in counts:
    decimal = int(output, 2)
    phase = decimal / (2**n_count)
    frac = Fraction(phase).limit_denominator(N)
    print(f"{output} -> phase={phase:.4f} -> {frac} -> r={frac.denominator}")

Counts: {'00000': 161, '11010': 142, '10010': 7, '01010': 28, '00001': 184, '11100': 4, '01011': 36, '10101': 92, '10100': 114, '11011': 123, '01101': 22, '01100': 36, '00110': 10, '00101': 7, '11000': 5, '00111': 12, '10111': 4, '00100': 7, '10011': 6, '00011': 2, '10110': 4, '11001': 2, '11101': 4, '11111': 1, '11110': 1, '01111': 2, '00010': 4, '10001': 1, '01000': 2, '01001': 1}

Phase estimation:
00000 -> phase=0.0000 -> 0 -> r=1
11010 -> phase=0.8125 -> 13/16 -> r=16
10010 -> phase=0.5625 -> 9/16 -> r=16
01010 -> phase=0.3125 -> 5/16 -> r=16
00001 -> phase=0.0312 -> 1/21 -> r=21
11100 -> phase=0.8750 -> 7/8 -> r=8
01011 -> phase=0.3438 -> 7/20 -> r=20
10101 -> phase=0.6562 -> 13/20 -> r=20
10100 -> phase=0.6250 -> 5/8 -> r=8
11011 -> phase=0.8438 -> 16/19 -> r=19
01101 -> phase=0.4062 -> 7/17 -> r=17
01100 -> phase=0.3750 -> 3/8 -> r=8
00110 -> phase=0.1875 -> 3/16 -> r=16
00101 -> phase=0.1562 -> 3/19 -> r=19
11000 -> phase=0.7500 -> 3/4 -> r=4
00111 -> phase=0.2188 -> 2/9 -> r=